# 확장 기능: 한글 글자 검출기 학습 (Google Colab)

1. 한글 합성 데이터 생성 (`hangul_dataset.py`): 검출 학습 이미지 + 떠다니는 한글 추적 영상
2. YOLO11n Fine-tuning → `models/hangul_text.pt` (+ 학습 설정·성능 `models/hangul_text.json`)
3. 결과를 Google Drive에 저장

**시작 전에**: **런타임 → 런타임 유형 변경 → T4 GPU**를 고르고 위에서부터 차례로 실행하세요.

## 0. GPU 확인과 Google Drive 연결

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 설정 — 필요하면 여기만 바꾸세요.
BRANCH = 'main'              # 아직 merge 전이면 'claude/hangul'
DRIVE = '/content/drive/MyDrive'
OUT = f'{DRIVE}/AI_Project_Level_3_outputs/hangul'
PROJECT = '/content/AI_Project_Level_3-main'
TRAIN_IMAGES = 6000          # 학습 이미지 수 (val·test는 각 500장)
USE_COCO_BACKGROUNDS = True  # COCO val2017 사진(약 1GB)을 배경으로 섞어 실제 장면과 비슷하게 만듭니다.
EPOCHS = 40
BATCH = 32                   # 메모리 부족(CUDA out of memory)이 나면 16

## 1. 코드·글꼴·패키지

In [ ]:
import os
if not os.path.isdir(PROJECT):
    !git clone -q -b {BRANCH} https://github.com/kumohyunho/AI_Project_Level_3-main.git {PROJECT}
%cd {PROJECT}
!git log --oneline -1
!apt-get -qq install -y fonts-nanum fonts-nanum-extra > /dev/null
!pip -q install ultralytics==8.3.233 scipy

## 2. (선택) 배경용 COCO val2017 사진

Drive에 한 번 받아 두면 다음부터 다시 받지 않습니다.

In [ ]:
BACKGROUNDS = ''
if USE_COCO_BACKGROUNDS:
    cache = f'{DRIVE}/datasets/val2017.zip'
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    if not os.path.isfile(cache):
        !wget -q --show-progress -O /content/val2017.zip http://images.cocodataset.org/zips/val2017.zip
        !cp /content/val2017.zip "{cache}"
    else:
        !cp "{cache}" /content/val2017.zip
    !mkdir -p datasets/raw/coco && unzip -q -o /content/val2017.zip -d datasets/raw/coco
    BACKGROUNDS = '--backgrounds datasets/raw/coco/val2017'
print(BACKGROUNDS or '배경: sample.mp4 + 무늬 배경만 사용')

## 3. 한글 합성 데이터 생성

`splits.json`에 사용한 글꼴(test 전용 글꼴 포함)과 이미지·박스 수가 기록됩니다.

In [ ]:
!rm -rf datasets/hangul
!python hangul_dataset.py --output datasets/hangul --train {TRAIN_IMAGES} {BACKGROUNDS}
!python -c "import json; d=json.load(open('datasets/hangul/splits.json')); print(json.dumps({k: d[k] for k in ('counts', 'fonts')}, ensure_ascii=False, indent=1))"

## 4. 검출기 Fine-tuning

끝나면 val·test mAP가 출력되고 `models/hangul_text.pt`와 학습 설정·성능 `models/hangul_text.json`이 설치됩니다.

In [ ]:
!python train_models.py hangul --data datasets/hangul/dataset.yaml --weights yolo11n.pt --epochs {EPOCHS} --batch {BATCH} --workers 4 --device 0 --install
!cat models/hangul_text.json

## 5. 결과 확인 (test 영상 한 프레임)

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
frame = 'datasets/hangul/tracking/test/hangul-01/img1/000060.jpg'
YOLO('models/hangul_text.pt').predict(frame, save=True, project='/content/preview', name='hangul', exist_ok=True, verbose=False)
display(Image('/content/preview/hangul/000060.jpg'))

## 6. Google Drive에 저장

`AI_Project_Level_3_outputs/hangul/`의 **`models/hangul_text.pt`와 `models/hangul_text.json`**을 내려받아
Claude 대화창에 올려 주세요. `claude/hangul` 브랜치에 커밋해 PR에 포함합니다.
`training_runs/hangul/`에는 학습 곡선(`results.png`)과 혼동 행렬이 있어 보고서에 쓸 수 있습니다.

In [ ]:
!mkdir -p "{OUT}/models" "{OUT}/training_runs"
!cp -v models/hangul_text.pt models/hangul_text.json "{OUT}/models/"
!cp -v datasets/hangul/splits.json "{OUT}/"
!rsync -a --exclude '*.cache' training_runs/hangul "{OUT}/training_runs/"